# RBC Contour Test — Detection Weights + Bbox-Cropped Contour Extraction

**Important caveat:** `best.pt` (from `rbc_yolo/yolo11n_1024`) is a **box-only** detector —
it was never trained with a mask head, so `results.masks` will always be `None` for it.
Ultralytics cannot produce learned segmentation contours from a detection-only checkpoint.

**What this notebook does instead:** uses the trained YOLO model to get fast, accurate
RBC *bounding boxes*, then extracts the *actual cell contour* inside each box with a
lightweight classical-CV step (HSV-saturation threshold + `cv2.findContours`, cropped to
the box — the ~54x-faster approach benchmarked earlier vs. running contour extraction on
the full frame). This gives a real per-cell polygon today, without waiting on a `-seg`
retrain.

**Pipeline:** YOLO detect (boxes) → crop each box → threshold + contour → polygon in
original image coordinates. Timing is measured separately for the detect step and the
contour step so you can see where the budget goes.

**Next step for learned (non-heuristic) contours:** train `yolo11n-seg.pt` on polygon
labels — see Section 9b of `rbc_detection.ipynb`.

In [6]:
import json, time
from pathlib import Path

import numpy as np
import cv2
from ultralytics import YOLO

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path('F:/Livo/Data - 2026/Rbc/rbc_yolo')
DATASET_DIR = Path('F:/Livo/Data - 2026/Rbc/yolo_dataset')
OUT_DIR     = Path('F:/Livo/Data - 2026/Rbc/rbc_contour_test_output')
BEST_PT     = PROJECT_DIR / 'yolo11n_1024' / 'weights' / 'best.pt'

OVERLAY_DIR = OUT_DIR / 'overlays'
JSON_OUT    = OUT_DIR / 'json'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(exist_ok=True)
JSON_OUT.mkdir(exist_ok=True)

# ── Inference / contour params ───────────────────────────────────────────────
IMGSZ       = 1024
CONF_THRES  = 0.25
IOU_THRES   = 0.45
BOX_PAD     = 3      # px padding around each bbox before thresholding
MIN_AREA_FR = 0.10   # discard contours smaller than this fraction of the crop area

# ── Size thresholds (pixel diameter at 100x magnification) ──────────────────
MICROCYTE_MAX = 35
MACROCYTE_MIN = 55

# ── 39-class definitions (shape x chromia) — same scheme as rbc_detection.ipynb
SHAPE_NAMES = [
    'acanthocytes', 'bite_cells', 'blister_cells', 'echinocytes', 'ovalocytes',
    'stomatocytes', 'schistocytes', 'sickle_cells', 'spherocytes', 'target_cells',
    'teardrop_cells', 'elliptocytes', 'normocytes',
]
CHROMIA_NAMES = ['hypochromic', 'normochromic', 'hyperchromic']
CLASSES       = [f'{s}_{c}' for s in SHAPE_NAMES for c in CHROMIA_NAMES]
ID_TO_SHAPE   = {i*3+j: SHAPE_NAMES[i]   for i in range(13) for j in range(3)}
ID_TO_CHROMIA = {i*3+j: CHROMIA_NAMES[j] for i in range(13) for j in range(3)}

SHAPE_COLOR = {
    'normocytes': (0,255,0), 'ovalocytes': (255,255,0), 'echinocytes': (0,165,255),
    'acanthocytes': (255,0,255), 'bite_cells': (0,255,255), 'blister_cells': (128,0,255),
    'schistocytes': (0,128,255), 'sickle_cells': (255,50,50), 'spherocytes': (50,255,100),
    'target_cells': (255,200,0), 'teardrop_cells': (200,0,255), 'stomatocytes': (0,200,200),
    'elliptocytes': (200,255,0),
}
DEFAULT_COLOR = (200, 200, 200)

print(f'BEST_PT   : {BEST_PT}  [{"EXISTS" if BEST_PT.exists() else "NOT FOUND"}]')
print(f'Output    : {OUT_DIR}')

BEST_PT   : F:\Livo\Data - 2026\Rbc\rbc_yolo\yolo11n_1024\weights\best.pt  [EXISTS]
Output    : F:\Livo\Data - 2026\Rbc\rbc_contour_test_output


In [7]:
t0 = time.perf_counter()
model = YOLO(str(BEST_PT))
load_ms = (time.perf_counter() - t0) * 1000
print(f'Model loaded in {load_ms:.1f} ms')

Model loaded in 44.8 ms


In [8]:
def compute_size(bbox_w, bbox_h):
    diam = (bbox_w + bbox_h) / 2.0
    if diam < MICROCYTE_MAX: return 'Microcyte'
    if diam > MACROCYTE_MIN: return 'Macrocyte'
    return 'Normocyte'


def contour_in_bbox(img_bgr, x1, y1, x2, y2, pad=BOX_PAD, min_area_fr=MIN_AREA_FR):
    """
    Extract the RBC's real pixel contour inside one YOLO box.
    Crops to the box (+pad) first, so cv2.findContours only ever scans a
    ~(box_w x box_h) region instead of the full frame — this is the ~54x
    speedup measured earlier vs. running contour extraction on full-size masks.
    Returns polygon points in ORIGINAL image pixel coordinates, or None.
    """
    H, W = img_bgr.shape[:2]
    cx1, cy1 = max(0, x1 - pad), max(0, y1 - pad)
    cx2, cy2 = min(W, x2 + pad), min(H, y2 + pad)
    crop = img_bgr[cy1:cy2, cx1:cx2]
    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    _, mask = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    crop_area = crop.shape[0] * crop.shape[1]
    best = max(contours, key=cv2.contourArea)
    if cv2.contourArea(best) < min_area_fr * crop_area:
        return None

    pts = best.reshape(-1, 2).astype(np.float32)
    pts[:, 0] += cx1   # back to original image coordinates
    pts[:, 1] += cy1
    return pts


def detect_and_contour(model, img_bgr):
    """Runs YOLO detection, then bbox-cropped contour extraction per instance.
    Returns (detections, timing_dict)."""
    H, W = img_bgr.shape[:2]

    t0 = time.perf_counter()
    results = model.predict(img_bgr, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, verbose=False)
    detect_ms = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    dets, det_id = [], 1
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = [round(float(v)) for v in box.xyxy[0]]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            bw, bh = x2 - x1, y2 - y1
            if bw <= 0 or bh <= 0:
                continue
            cid = int(box.cls[0])
            pts = contour_in_bbox(img_bgr, x1, y1, x2, y2)
            det = {
                'id':         det_id,
                'bbox':       {'x': x1, 'y': y1, 'width': bw, 'height': bh},
                'class_id':   cid,
                'label':      CLASSES[cid] if cid < 39 else 'unknown',
                'shape':      ID_TO_SHAPE.get(cid, 'unknown'),
                'chromia':    ID_TO_CHROMIA.get(cid, 'unknown'),
                'size':       compute_size(bw, bh),
                'confidence': round(float(box.conf[0]), 4),
            }
            if pts is not None:
                det['contour'] = {
                    'points_px':   [[round(float(px), 1), round(float(py), 1)] for px, py in pts],
                    'points_norm': [[round(float(px) / W, 6), round(float(py) / H, 6)] for px, py in pts],
                }
            dets.append(det)
            det_id += 1
    contour_ms = (time.perf_counter() - t1) * 1000

    timing = {'detect_ms': round(detect_ms, 2), 'contour_ms': round(contour_ms, 2),
              'total_ms': round(detect_ms + contour_ms, 2)}
    return dets, timing


def draw_contour_overlay(img_bgr, detections):
    vis = img_bgr.copy()
    overlay = img_bgr.copy()
    for det in detections:
        color = SHAPE_COLOR.get(det['shape'], DEFAULT_COLOR)
        contour = det.get('contour')
        if contour and contour['points_px']:
            pts = np.array(contour['points_px'], dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], color)
            cv2.polylines(vis, [pts], isClosed=True, color=color, thickness=1, lineType=cv2.LINE_AA)
        else:
            b = det['bbox']
            cv2.rectangle(vis, (b['x'], b['y']), (b['x']+b['width'], b['y']+b['height']), color, 1)
    return cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)

print('Helpers ready: contour_in_bbox, detect_and_contour, draw_contour_overlay')

Helpers ready: contour_in_bbox, detect_and_contour, draw_contour_overlay


In [9]:
TEST_IMAGES = sorted((DATASET_DIR / 'images' / 'test').glob('*.jpg'))
N_RUN       = min(20, len(TEST_IMAGES))   # cap for a quick timing pass; raise to test all 118
TARGET_MS   = 200

print(f'Found {len(TEST_IMAGES)} test images — running timing pass on first {N_RUN}\n')

timing_log = []
for idx, img_path in enumerate(TEST_IMAGES[:N_RUN], 1):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f'  [{idx:3d}] SKIP (unreadable): {img_path.name}')
        continue

    dets, timing = detect_and_contour(model, img)
    n_contours = sum(1 for d in dets if 'contour' in d)
    timing_log.append({'image': img_path.name, 'n_dets': len(dets), 'n_contours': n_contours, **timing})

    with open(JSON_OUT / (img_path.stem + '_pred.json'), 'w') as f:
        json.dump({'image_id': img_path.name, 'detections': dets, 'timing': timing}, f, indent=2)

    overlay = draw_contour_overlay(img, dets)
    cv2.imwrite(str(OVERLAY_DIR / (img_path.stem + '_overlay.jpg')), overlay)

    flag = '' if timing['total_ms'] < TARGET_MS else '  <-- OVER 200ms BUDGET'
    print(f"  [{idx:3d}/{N_RUN}] {img_path.name:<22s}  {len(dets):3d} dets  "
          f"{n_contours:3d} contours  detect={timing['detect_ms']:6.1f}ms  "
          f"contour={timing['contour_ms']:6.1f}ms  total={timing['total_ms']:6.1f}ms{flag}")

# ── Summary (exclude cold-start image 1) ──────────────────────────────────────
rows = timing_log[1:] if len(timing_log) > 1 else timing_log
detect_arr  = np.array([r['detect_ms']  for r in rows])
contour_arr = np.array([r['contour_ms'] for r in rows])
total_arr   = np.array([r['total_ms']   for r in rows])

print(f'\n{"="*70}')
print(f'  TIMING SUMMARY  (n={len(rows)}, excl. cold start, target <{TARGET_MS}ms/FOV)')
print(f'{"="*70}')
print(f'  {"Stage":<12s} {"Median":>8s} {"Mean":>8s} {"Min":>8s} {"Max":>8s}')
for name, arr in [('Detect', detect_arr), ('Contour', contour_arr), ('Total', total_arr)]:
    print(f'  {name:<12s} {np.median(arr):7.1f}ms {arr.mean():7.1f}ms {arr.min():7.1f}ms {arr.max():7.1f}ms')

pct_under = (total_arr < TARGET_MS).mean() * 100
print(f'\n  {pct_under:.1f}% of FOVs under {TARGET_MS}ms  (avg {np.mean([r["n_dets"] for r in rows]):.0f} RBCs/FOV)')

with open(OUT_DIR / 'timing_stats.json', 'w') as f:
    json.dump(timing_log, f, indent=2)
print(f'\n  Overlays -> {OVERLAY_DIR}')
print(f'  JSONs    -> {JSON_OUT}')
print(f'  Stats    -> {OUT_DIR}/timing_stats.json')

Found 118 test images — running timing pass on first 20

  [  1/20] Img_8_0.jpg              32 dets   28 contours  detect=2862.6ms  contour= 969.0ms  total=3831.6ms  <-- OVER 200ms BUDGET
  [  2/20] Img_8_1.jpg              44 dets   38 contours  detect=  21.7ms  contour=  41.7ms  total=  63.4ms
  [  3/20] Img_8_10.jpg             45 dets   39 contours  detect=  16.4ms  contour=  38.1ms  total=  54.5ms
  [  4/20] Img_8_11.jpg             55 dets   49 contours  detect=  16.7ms  contour= 145.5ms  total= 162.2ms
  [  5/20] Img_8_12.jpg             54 dets   49 contours  detect=  18.1ms  contour=  44.1ms  total=  62.2ms
  [  6/20] Img_8_13.jpg             47 dets   42 contours  detect=  18.9ms  contour=  43.9ms  total=  62.8ms
  [  7/20] Img_8_14.jpg             50 dets   42 contours  detect=  16.3ms  contour=  42.3ms  total=  58.6ms
  [  8/20] Img_8_15.jpg             40 dets   31 contours  detect=  21.3ms  contour=  46.1ms  total=  67.4ms
  [  9/20] Img_8_16.jpg             69 dets   61

## Notes

- **Model:** `rbc_yolo/yolo11n_1024/weights/best.pt` — box-only detection, mAP50=0.520.
- **Contours are heuristic, not learned:** HSV-saturation Otsu threshold + `cv2.findContours`,
  run inside each YOLO box (not full-frame) — this is what keeps it fast. Central pallor
  and staining variance can affect edge accuracy on some cells; inspect `overlays/` to
  judge whether it's "exact" enough for your use case.
- **If accuracy isn't good enough:** the fix is a learned `-seg` model (Section 9b of
  `rbc_detection.ipynb`), not tuning this threshold further — Otsu-on-saturation will not
  match a trained mask's precision on irregular shapes (sickle cells, schistocytes, etc.).
- Re-run the timing cell with `N_RUN = len(TEST_IMAGES)` to check all 118 test FOVs instead
  of the first 20.